# 0.0 - Imports

In [1]:
import pandas as pd
import requests
import time

### 0.1 - Helper Functions

In [2]:
def teste_busca(codigo_grupo, codigo_classe, tamanho_pagina=500):
    """
    Busca todos os itens de contratação de uma classe de material,
    percorrendo todas as páginas disponíveis.
    """
    todos_itens = []
    pagina = 1

    while True:
        params = {
            "materialOuServico": "M",
            "codigoGrupo": codigo_grupo,
            "codigoClasse": codigo_classe,
            "pagina": pagina,
            "tamanhoPagina": tamanho_pagina,
        }
        resposta = requests.get(url_itens, params=params)
        dados = resposta.json()

        todos_itens.extend(dados["resultado"])

        print(f"Página {pagina}/{dados['totalPaginas']} — "
              f"{len(dados['resultado'])} itens coletados nesta página")

        if pagina >= dados["totalPaginas"]:
            break

        pagina += 1
        time.sleep(0.3)  # pequena pausa entre chamadas, boa prática

    return todos_itens

### 0.2 - Validação da API

#### 0.2.1 - Teste de API - Requisição e Funcionamento

In [3]:
# url da API
url = "https://dadosabertos.compras.gov.br/modulo-material/1_consultarGrupoMaterial"

# armazenamento da requisição
resposta_requisição = requests.get(url)

#resultado da requisição: 200 - OK; Outro resultado: Erro de Requisição - Investigar
print("Status code:", resposta_requisição.status_code)

Status code: 200


#### 0.2.2 - Verificação dos Dados da API

In [4]:
# converte o JSON da resposta em dict/list do Python
dados = resposta_requisição.json()  

#Mostra os resultados da verificação da API
print(type(dados))
print(dados.keys())
dados["resultado"][0]

<class 'dict'>
dict_keys(['resultado', 'totalRegistros', 'totalPaginas', 'paginasRestantes'])


{'codigoGrupo': 10,
 'nomeGrupo': 'ARMAMENTO',
 'statusGrupo': True,
 'dataHoraAtualizacao': '2021-10-16T09:16:33.723625'}

#### 0.2.3 - Teste de conexao com parametros

In [5]:
#URL para consulta dos itens
url_itens = "https://dadosabertos.compras.gov.br/modulo-material/4_consultarItemMaterial"

#Paramentros para a função requests
params = {
    "materialOuServico": "M",       # M = Material (não Serviço)
    "codigoGrupo": 61,              # Condutores Elétricos e Equip. p/ Geração e Distribuição
    "codigoClasse": 6145,           # Fios e Cabos Elétricos
    "pagina": 1,
    "tamanhoPagina": 20,            # Teste
}

#Armazena o request da API na variavel Resposta
resposta = requests.get(url_itens, params=params)

#Print da requisição realizada no site
print("Status code:", resposta.status_code)
print("URL final montada:", resposta.url)
print("Status:", resposta.status_code)
print("Total registros:", dados.get("totalRegistros"))

#Trasformação da varavel resposta de JSON para Dicionário
dados_itens = resposta.json()

Status code: 200
URL final montada: https://dadosabertos.compras.gov.br/modulo-material/4_consultarItemMaterial?materialOuServico=M&codigoGrupo=61&codigoClasse=6145&pagina=1&tamanhoPagina=20
Status: 200
Total registros: 79


#### 0.2.4 - Teste conversão JSON para DataFrame

In [6]:
#Transformação em Dataset para Teste
df_teste = pd.DataFrame(dados_itens["resultado"])
df_teste.head(3)

,codigoItem,codigoGrupo,nomeGrupo,codigoClasse,nomeClasse,codigoPdm,nomePdm,descricaoItem,statusItem,itemSustentavel,codigo_ncm,descricao_ncm,aplica_margem_preferencia,dataHoraAtualizacao
0,217430,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE EST...",True,False,None,None,None,2021-10-16T09:43:08.030221
1,217429,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE EST...",True,False,None,None,None,2021-10-16T09:43:08.030221
2,261051,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE , R...",True,False,None,None,None,2021-10-16T09:43:08.030221


#### 0.2.5 - Validação da Função para Extração de Dados

In [12]:
itens_6145 = teste_busca(codigo_grupo=61, codigo_classe=6145)
print(f"\nTotal coletado: {len(itens_6145)} itens")

df_6145 = pd.DataFrame(itens_6145)
df_6145.head()

Página 1/7 — 500 itens coletados nesta página
Página 2/7 — 500 itens coletados nesta página
Página 3/7 — 500 itens coletados nesta página
Página 4/7 — 500 itens coletados nesta página
Página 5/7 — 500 itens coletados nesta página
Página 6/7 — 500 itens coletados nesta página
Página 7/7 — 256 itens coletados nesta página

Total coletado: 3256 itens


,codigoItem,codigoGrupo,nomeGrupo,codigoClasse,nomeClasse,codigoPdm,nomePdm,descricaoItem,statusItem,itemSustentavel,codigo_ncm,descricao_ncm,aplica_margem_preferencia,dataHoraAtualizacao
0,217430,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE EST...",True,False,None,None,None,2021-10-16T09:43:08.030221
1,217429,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE EST...",True,False,None,None,None,2021-10-16T09:43:08.030221
2,261051,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE , R...",True,False,None,None,None,2021-10-16T09:43:08.030221
3,320738,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE , R...",True,False,None,None,None,2021-10-16T09:43:08.030221
4,320643,61,CONDUTORES ELÉTRICOS E EQUIPAMENTOS PARA GERAÇ...,6145,FIOS E CABOS ELÉTRICOS,606,CABINHO ELÉTRICO FLEXÍVEL,"CABINHO ELÉTRICO FLEXÍVEL, MATERIAL: COBRE , R...",True,False,None,None,None,2021-10-16T09:43:08.030221
